Завантаження даних та підготовка мультимодальності

In [1]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath(os.path.join('..')))

from src.extraction import SignalExtractor, TextExtractor
from src.features import FeatureInteractionTransformer, OutofFoldWoEEncoder
from src.selection import three_level_feature_selection

# 1. Завантажуємо табличний датасет з ЛР №1
df = pd.read_csv('cs-training.csv').iloc[:5000].drop(columns=['Unnamed: 0'], errors='ignore')

# 2. Створюємо неструктуровані колонки:
# - Текстовий коментар: якщо у клієнта були прострочення > 2 разів, пишемо "poor credit history", інакше "good client status"
df['text_comments'] = ["poor credit history" if x > 2 else "good client status" 
                       for x in df['NumberOfTime30-59DaysPastDueNotWorse']]

# - Сигнальна історія (транзакції): створюємо список із 3 чисел на основі використання кредитного ліміту
df['signal_history'] = [[x * 10, x * 5, x * 2] 
                         for x in df['RevolvingUtilizationOfUnsecuredLines']]

# 3. Викликаємо екстрактор для тексту (TF-IDF + SVD) -> повертає 5 нових колонок text_svd_0..4
text_ext = TextExtractor()
text_features = text_ext.fit_transform(df['text_comments'])

# 4. Викликаємо екстрактор для сигналу -> повертає 3 нові колонки (mean, std, max)
signal_ext = SignalExtractor()
signal_features = signal_ext.fit_transform(df['signal_history'])

# 5. Склеюємо початкову таблицю та нові отримані числові колонки в одну датафрейм-матрицю
df_rich = pd.concat(
    [df.reset_index(drop=True), text_features, signal_features], axis=1
)

print("Успішно завантажено та збагачено дані!")
print("Початкова кількість колонок:", df.shape[1])
print("Фінальна розмірність df_rich:", df_rich.shape)

Успішно завантажено та збагачено дані!
Початкова кількість колонок: 13
Фінальна розмірність df_rich: (5000, 21)


In [2]:
display(df_rich)

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,...,text_comments,signal_history,text_svd_0,text_svd_1,text_svd_2,text_svd_3,text_svd_4,signal_mean,signal_std,signal_max
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,...,good client status,"[7.66126609, 3.830633045, 1.532253218]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,4.341384,2.528089,7.661266
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,...,good client status,"[9.57151019, 4.785755095, 1.914302038]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,5.423856,3.158437,9.571510
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,...,good client status,"[6.5818014, 3.2909007, 1.31636028]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,3.729687,2.171884,6.581801
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,...,good client status,"[2.33809776, 1.16904888, 0.467619552]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,1.324922,0.771533,2.338098
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,...,good client status,"[9.072394000000001, 4.5361970000000005, 1.8144...",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,5.141023,2.993737,9.072394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0,0.516205,54,1,0.274175,4456.0,7,0,0,0,...,good client status,"[5.16205291, 2.581026455, 1.032410582]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,2.925163,1.703391,5.162053
4996,0,0.153128,32,1,0.082167,5500.0,6,0,0,0,...,good client status,"[1.53128229, 0.765641145, 0.306256458]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,0.867727,0.505297,1.531282
4997,0,0.413590,55,0,1.205276,1212.0,9,0,1,0,...,good client status,"[4.13589581, 2.067947905, 0.827179162]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,2.343674,1.364776,4.135896
4998,0,0.273258,44,0,3.762905,1142.0,5,0,2,0,...,good client status,"[2.73258139, 1.366290695, 0.546516278]",1.0,-1.602469e-16,-5.551115e-17,0.0,6.162976e-33,1.548463,0.901706,2.732581


AutoFE за допомогою Featuretools (DFS)

In [3]:
import featuretools as ft

# 1. Вилучаємо цільову змінну SeriousDlqin2yrs для безпечного AutoFE
df_for_dfs = df_rich.drop(columns=['SeriousDlqin2yrs'], errors='ignore')

# 2. Створюємо EntitySet та запускаємо DFS
es = ft.EntitySet(id="credit_data")
es = es.add_dataframe(
    dataframe_name="clients",
    dataframe=df_for_dfs.reset_index(drop=True).reset_index(),
    index="index",
)

# 3. Запускаємо Deep Feature Synthesis (DFS)
# Використовуємо арифметичні трансформери для створення комбінацій
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="clients",
    max_depth=1,
    trans_primitives=[
        "divide_numeric",
        "multiply_numeric",
        "add_numeric",
        "absolute",
    ],
)

n_before = df_rich.shape[1]
n_after = feature_matrix.shape[1]

print(f"Кількість ознак ДО DFS: {n_before}")
print(
    f"Кількість ознак ПІСЛЯ DFS: {n_after} (збільшення у {n_after / n_before:.1f} разів!)"
)

d:\Python\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
d:\Python\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
d:\Python\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
d:\Python\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected,

Кількість ознак ДО DFS: 21
Кількість ознак ПІСЛЯ DFS: 649 (збільшення у 30.9 разів!)


Очищення матриці X (CACE Check + Inf/NaN/Category)

In [4]:
# 1. CACE Check: Видаляємо вік age та будь-які його похідні
protected_attr = 'age'
features_to_use = [
    c
    for c in feature_matrix.columns
    if c != protected_attr
    and not c.startswith('age_')
    and not c.endswith('_age')
    and '_age_' not in c
    and 'SeriousDlqin2yrs' not in c
]

X = feature_matrix[features_to_use].copy()
y = df['SeriousDlqin2yrs'].values
age_group = (df['age'] > 50).astype(int)  # Для метрик справедливості

# 2. Очищення категорій, inf та NaN
for col in X.columns:
    if str(X[col].dtype) in ['category', 'object']:
        X[col] = X[col].astype('category').cat.codes

X = X.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)

print(
    "Матриця X успішно очищена від inf, NaN та категорій! Фінальна розмірність:",
    X.shape,
)

Матриця X успішно очищена від inf, NaN та категорій! Фінальна розмірність: (5000, 648)


Оголошення функції для розрахунку справедливості

In [5]:
def calculate_fairness_metrics(y_true, y_pred, group):
    """Обчислює Demographic Parity Ratio (DPR) та Equalized Odds Ratio (EOR)"""
    rate_g0 = np.mean(y_pred[group == 0])
    rate_g1 = np.mean(y_pred[group == 1])
    dpr = (
        min(rate_g0 / rate_g1, rate_g1 / rate_g0)
        if (rate_g0 > 0 and rate_g1 > 0)
        else 0.0
    )

    tpr_g0 = recall_score(y_true[group == 0], y_pred[group == 0], zero_division=0)
    tpr_g1 = recall_score(y_true[group == 1], y_pred[group == 1], zero_division=0)
    eor = (
        min(tpr_g0 / tpr_g1, tpr_g1 / tpr_g0)
        if (tpr_g0 > 0 and tpr_g1 > 0)
        else 0.0
    )

    return dpr, eor

Експеримент 0

In [6]:
# ==============================================================================
# ЕКСПЕРИМЕНТ 0: Non-ML Heuristic (Правило з Лаби 1)
# ==============================================================================
y_pred_0 = (
    (df['MonthlyIncome'] < 1000) | (df['NumberOfTimes90DaysLate'] > 0)
).astype(int)
auc_exp0 = roc_auc_score(y, y_pred_0)
print(f"ROC-AUC Exp 0: {auc_exp0:.4f}")

ROC-AUC Exp 0: 0.6375


In [7]:
# --- ЄДИНИЙ ЧЕСНИЙ СПЛІТ 70/30 ---
X_train, X_test, y_train, y_test, age_tr, age_te = train_test_split(
    X, y, age_group, test_size=0.3, random_state=42, stratify=y
)

Експеримент 1

In [8]:
# ==============================================================================
# ЕКСПЕРИМЕНТ 1: Baseline (Оригінальні колонки)
# ==============================================================================
orig_cols = [
    c
    for c in df.columns
    if c not in ['SeriousDlqin2yrs', 'age', 'text_comments', 'signal_history']
]
rf_exp1 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_exp1.fit(X_train[orig_cols], y_train)

probs_exp1 = rf_exp1.predict_proba(X_test[orig_cols])[:, 1]
preds_exp1 = (probs_exp1 > 0.5).astype(int)
auc_exp1 = roc_auc_score(y_test, probs_exp1)
dpr_exp1, eor_exp1 = calculate_fairness_metrics(y_test, preds_exp1, age_te)

print(f"ROC-AUC Exp 1: {auc_exp1:.4f}")

ROC-AUC Exp 1: 0.8339


Експеримент 2

In [9]:
# ==============================================================================
# ЕКСПЕРИМЕНТ 2: Manual FE (Кастомні взаємодії з Кроку 1)
# ==============================================================================
interaction_trans = FeatureInteractionTransformer()
X_tr_m = interaction_trans.fit_transform(X_train[orig_cols])
X_te_m = interaction_trans.transform(X_test[orig_cols])

rf_exp2 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_exp2.fit(X_tr_m, y_train)
auc_exp2 = roc_auc_score(y_test, rf_exp2.predict_proba(X_te_m)[:, 1])

print(f"ROC-AUC Exp 2: {auc_exp2:.4f}")

ROC-AUC Exp 2: 0.8380


Експеримент 3

In [10]:
# ЧЕСНИЙ ВІДБІР: проводиться виключно на X_train і y_train
selected_cols = three_level_feature_selection(
    X_train, y_train, k_filter=30, k_wrapper=15
)

rf_exp3 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_exp3.fit(X_train[selected_cols], y_train)

auc_exp3 = roc_auc_score(y_test, rf_exp3.predict_proba(X_test[selected_cols])[:, 1])

print(f"ROC-AUC Експерименту 3: {auc_exp3:.4f}")
print("Обрані чесні колонки:", selected_cols)

d:\Python\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


ROC-AUC Експерименту 3: 0.7020
Обрані чесні колонки: ['NumberOfTimes90DaysLate / NumberOfOpenCreditLinesAndLoans', 'signal_max / NumberOfTime60-89DaysPastDueNotWorse', 'age / NumberOfTime60-89DaysPastDueNotWorse']


Експеримент 4

In [15]:
# ==============================================================================
# ЕКСПЕРИМЕНТ 4: Leaky Pipeline (ПАСТКА ВИТОКУ ДАНИХ)
# ==============================================================================
# РЯДОК ВИТОКУ №1: Target Encoding / OOF кодування робиться на ВСЬОМУ датасеті до спліту!
X_leaky = X.copy()
X_leaky['target_leaky_enc'] = df.groupby('NumberOfTimes90DaysLate')[
    'SeriousDlqin2yrs'
].transform('mean')

# РЯДОК ВИТОКУ №2: Відбір колонок запускається на всьому X_leaky і y ДО train_test_split
leaky_selected_cols = three_level_feature_selection(
    X_leaky, y, k_filter=30, k_wrapper=15
)

X_tr_l, X_te_l, y_tr_l, y_te_l, age_tr_l, age_te_l = train_test_split(
    X_leaky[leaky_selected_cols],
    y,
    age_group,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

rf_exp4 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_exp4.fit(X_tr_l, y_tr_l)

probs_exp4 = rf_exp4.predict_proba(X_te_l)[:, 1]
preds_exp4 = (probs_exp4 > 0.5).astype(int)
auc_exp4 = roc_auc_score(y_te_l, probs_exp4)
dpr_exp4, eor_exp4 = calculate_fairness_metrics(y_te_l, preds_exp4, age_te_l)

print(f"ROC-AUC Експерименту 4 (Leaky Pipeline): {auc_exp4:.4f}")

d:\Python\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


ROC-AUC Експерименту 4 (Leaky Pipeline): 0.7040


In [12]:
# ==============================================================================
# ЗВЕДЕНА ТАБЛИЦЯ РЕЗУЛЬТАТІВ
# ==============================================================================
results_df = pd.DataFrame({
    'Експеримент': [
        '0. Non-ML Heuristic',
        '1. Baseline',
        '2. Manual FE',
        '3. Extracted + AutoFE',
        '4. Leaky Pipeline',
    ],
    'Опис': [
        'Евристичне правило з Лаби 1',
        'Вихідний датасет (без нових ознак)',
        'Кастомні взаємодії з Кроку 1',
        'Мультимодальність + DFS + Чесний відбір',
        'Відбір ознак глобально до Train/Test split',
    ],
    'ROC-AUC': [auc_exp0, auc_exp1, auc_exp2, auc_exp3, auc_exp4],
})

print("====================================================================")
print("                  ПІДСУМКОВІ РЕЗУЛЬТАТИ МОДЕЛЮВАННЯ                  ")
print("====================================================================")
display(results_df)

print("\n--------------------------------------------------------------------")
print("                  ПОРІВНЯННЯ МЕТРИК СПРАВЕДЛИВОСТІ                   ")
print("--------------------------------------------------------------------")
fairness_df = pd.DataFrame({
    'Експеримент': ['1. Baseline (Чесний)', '4. Leaky Pipeline (Витік)'],
    'DPR (Demographic Parity)': [dpr_exp1, dpr_exp4],
    'EOR (Equalized Odds)': [eor_exp1, eor_exp4],
})
display(fairness_df)

                  ПІДСУМКОВІ РЕЗУЛЬТАТИ МОДЕЛЮВАННЯ                  


,Експеримент,Опис,ROC-AUC
0,0. Non-ML Heuristic,Евристичне правило з Лаби 1,0.637520
1,1. Baseline,Вихідний датасет (без нових ознак),0.833948
2,2. Manual FE,Кастомні взаємодії з Кроку 1,0.838043
3,3. Extracted + AutoFE,Мультимодальність + DFS + Чесний відбір,0.701971
4,4. Leaky Pipeline,Відбір ознак глобально до Train/Test split,0.704010



--------------------------------------------------------------------
                  ПОРІВНЯННЯ МЕТРИК СПРАВЕДЛИВОСТІ                   
--------------------------------------------------------------------


,Експеримент,DPR (Demographic Parity),EOR (Equalized Odds)
0,1. Baseline (Чесний),0.657820,0.826667
1,4. Leaky Pipeline (Витік),0.823427,0.206667
